# 03. Stationarity and Decomposition

В этом ноутбуке разбираем структуру ряда, делаем декомпозицию и проверяем стационарность ADF-тестом.

## 1. Load processed data

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.visualization import save_plot

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
IMAGES = PROJECT_ROOT / "images"
IMAGES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PROCESSED / "daily_sales_series.csv", parse_dates=["date"])
series = df.set_index("date").asfreq("D")["sales_clean"].fillna(0)

print(series.shape)
display(series.head())

## 2. Time series decomposition

Используем период `7`, чтобы выделить недельную сезонность.

In [ ]:
decomposition = seasonal_decompose(series, model="additive", period=7)
fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.suptitle("Time Series Decomposition", y=1.02)
save_plot(IMAGES / "decomposition.png")
plt.show()

## 3. ADF test

In [ ]:
def run_adf_test(values, series_name):
    result = adfuller(pd.Series(values).dropna(), autolag="AIC")
    statistic = result[0]
    p_value = result[1]
    interpretation = "stationary" if p_value < 0.05 else "non-stationary"
    print(f"{series_name}")
    print(f"ADF statistic: {statistic:.4f}")
    print(f"p-value: {p_value:.4f}")
    print(f"Interpretation: series is {interpretation} at alpha=0.05")
    return {"series": series_name, "adf_statistic": statistic, "p_value": p_value, "interpretation": interpretation}

adf_original = run_adf_test(series, "Original sales_clean")

## 4. Log transformation

In [ ]:
log_series = np.log1p(series)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(log_series.index, log_series.values, label="log1p sales_clean")
ax.set_title("Log-transformed Sales")
ax.set_xlabel("date")
ax.set_ylabel("log1p sales_clean")
ax.legend()
ax.grid(True)
plt.show()

## 5. Differencing

In [ ]:
diff_series = log_series.diff().dropna()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(diff_series.index, diff_series.values, label="first difference of log1p sales")
ax.set_title("Differenced Log Sales")
ax.set_xlabel("date")
ax.set_ylabel("diff log1p sales")
ax.legend()
ax.grid(True)
plt.show()

adf_diff = run_adf_test(diff_series, "First difference of log1p sales_clean")

## 6. Stationarity conclusions

Интерпретация ADF-теста:

- если `p-value < 0.05`, ряд считается стационарным на уровне значимости 5%;
- если `p-value >= 0.05`, нет оснований отвергнуть гипотезу о нестационарности.

Результаты этого ноутбука используются для обоснования SARIMA-модели с дифференцированием и недельной сезонностью.